# GridNet-HD — ข้อมูลของรูป qualitative: prediction รอบเสาไฟฟ้า ภายใต้ pose error (GPU, session สั้น)

รูปใน paper ที่ยังขาด: ให้ผู้อ่าน **เห็น** ว่าเมื่อ pose ของกล้องผิด head ระดับ feature ที่เทรนด้วย pose ที่ให้มา ทำสายไฟ/สายยึดหายไปตรงไหน และ head ตัวเดียวกันที่เทรนด้วย
pose-noise augmentation ไม่เป็น — notebook นี้ไม่ได้เทรนอะไร และไม่ได้ให้ตัวเลขใหม่ของ paper: มันรัน robustness protocol **เฉพาะ draw 0** ของระดับ error ที่เลือก
(`LEVELS`) แล้วเก็บ prediction ของจุดรอบเสาไฟฟ้าที่ใหญ่ที่สุด `PYLONS` ต้นของแต่ละโซน (รัศมี `RADIUS` ม.) เป็นไฟล์เล็ก ๆ ไว้บน Drive

- กล้องของ draw 0 ถูก seed ด้วย (ชื่อ sweep, ระดับ, ลำดับ draw) → เป็น **กล้องชุดเดียวกับ draw 0 ของ run `exact_v1`** ที่ตัวเลขใน paper มาจาก
  cell 7 เทียบ confusion matrix ของสอง run ให้เลย (ต้องต่างกัน 0.00) = รูปแสดง prediction ที่อยู่ในตัวเลขของ benchmark จริง ไม่ใช่รันแยก
- ต้องมีบน Drive แล้ว (มีครบจาก lab ก่อน ๆ): LiDAR cache + zone tar ของ val zones, feature bank (`GridNet/protocol/subsets_<RUN2D>/<zone>/subset.npz`),
  `GridNet/fusion/<head>/best.pth` ของ heads ใน `HEADS`
- **Runtime: GPU** — A100 คือเครื่องที่โหมด exact เคยรันจริง (L4 ยังไม่เคยลองกับโหมด exact; ถ้าได้แค่ L4 แล้ว memory ไม่พอ ให้ตั้ง `MODE = 'fast'`
  ซึ่งสำหรับ fusion heads ต่างจาก exact ไม่เกิน 0.04 mIoU)
- เวลาที่วัดได้ (14 พ.ย. 2026, A100): 21 นาทีทั้ง notebook (protocol 7.5 นาที)
- **ก่อนรัน**: `GridNet/code/gridnet_code.zip` บน Drive ต้องเป็นของ **update38 ขึ้นไป** (cell 4 ตรวจให้)
- **ผลลัพธ์**: `GridNet/qualitative/<TAG>/<zone>_crop<i>.npz` (ไฟล์ละไม่กี่ MB) + `preview/*.png` + `crops.txt` — เสร็จแล้วผลอยู่บน Drive (ใช้เลือกเสา/มุมมองและทำรูปของ paper)

**Resume**: Run all ซ้ำได้ — โซนที่เสร็จแล้วถูกข้าม แต่ถ้าเปลี่ยน `HEADS` / `LEVELS` / `RADIUS` / `PYLONS` / `MODE` ให้ตั้ง `TAG` ใหม่ (script หยุดเองถ้าไม่ตรง)

In [ ]:
#@title 1. GPU (the A100 is the configuration the exact mode has run on; nothing is trained here)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.get_device_capability()[0] >= 8, "need an L4 or A100 runtime (the environment tarball is built for these GPUs)"


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
TAG = 'qual_v1'                                  #@param {type:"string"}
HEADS = 'late_nat_b1,late_a,concat_n,concat_a,win5_n,win5_a'   #@param {type:"string"}
ZONES = 't1z5b,t1z8,t3z4,t4z1,t5b1,t5b5'         #@param {type:"string"}
LEVELS = 'trans:0,trans:0.1,trans:0.5,rot:1'     #@param {type:"string"}
PYLONS = 2                                       #@param {type:"integer"}
RADIUS = 35.0                                    #@param {type:"number"}
MODE = 'exact'                                   #@param ["exact", "fast"]
SEED = 0                                         #@param {type:"integer"}
REFERENCE_RUN = 'exact_v1'                       #@param {type:"string"}
PREVIEW_MODELS = 'concat_n,concat_a'             #@param {type:"string"}
PREVIEW_LEVELS = 'trans:0,trans:0.5,rot:1'       #@param {type:"string"}
RUN2D = 'upernet_convnext_t_v2'                  #@param {type:"string"}
SHUTDOWN_WHEN_DONE = False                       #@param {type:"boolean"}
CACHE_DRIVE = f'{DRIVE}/cache'
CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN2D}'
ASSOC_DIR = f'{DRIVE}/fusion/assoc_{RUN2D}'
SUBSET_DIR = f'{DRIVE}/protocol/subsets_{RUN2D}'
OUT = f'{DRIVE}/qualitative/{TAG}'
os.makedirs(OUT, exist_ok=True)
zones = [z.strip() for z in ZONES.split(',') if z.strip()]
heads = {}
for name in [h.strip() for h in HEADS.split(',') if h.strip()]:
    path = f'{DRIVE}/fusion/{name}/best.pth'
    assert os.path.exists(path), f"no checkpoint for head {name}: {path}"
    heads[name] = path
sweeps = {}                                      # the levels of the run: the wanted ones, and the delivered poses (0) in every sweep
for item in [x.strip() for x in LEVELS.split(',') if x.strip()]:
    sw, lv = item.split(':'); assert sw in ('trans', 'rot', 'drift'), f"unknown sweep in LEVELS: {item}"
    sweeps.setdefault(sw, [0.0])
    if float(lv) not in sweeps[sw]: sweeps[sw].append(float(lv))
for sw in sweeps: sweeps[sw].sort()
for z in zones:
    assert os.path.exists(f'{CACHE_DRIVE}/{z}/points.npz'), f"no LiDAR cache on Drive for {z}"
    assert os.path.exists(f'{SUBSET_DIR}/{z}/subset.npz'), f"no feature bank on Drive for {z}: {SUBSET_DIR}/{z}/subset.npz (colab_heads_lab writes them)"
done = []
if os.path.exists(f'{OUT}/protocol/results.json'):
    import json
    done = sorted({r['zone'] for r in json.load(open(f'{OUT}/protocol/results.json'))['runs']})
print("heads:", list(heads)); print("levels of the run:", sweeps, "| kept for the figure:", LEVELS)
print(f"{len(zones)} zones, {PYLONS} pylons each, radius {RADIUS:g} m, {MODE} mode  ->  {OUT}")
print(f"{len(done)}/{len(zones)} zones already on Drive:", done)


In [ ]:
#@title 3. Environment (restore tarball from Drive)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    assert os.path.exists(TAR), "no env tarball on Drive: run colab_train_litept cell 3 once"
    subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
PY = f'{ENV}/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pillow', 'matplotlib'], check=True)
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code + LitePT
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
assert os.path.exists('/content/code/gridnet/qualitative.py'), "gridnet_code.zip on Drive is older than update38: pack and upload it again"
if not os.path.exists('/content/LitePT'):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/prs-eth/LitePT.git", "/content/LitePT"], check=True)
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/LitePT'); sys.path.insert(0,'/content/code'); "
                    "from litept.model import LitePT; import gridnet.train.protocol, gridnet.qualitative; print('imports ok')"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])
assert 'imports ok' in r.stdout


In [ ]:
#@title 5. Data -> local disk: LiDAR caches, feature banks, 2D caches (zone tars), camera files of ZONES
import os, sys, shutil, subprocess, time
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, '/content/code')
from gridnet.train.maps2d import cached_image_ids, zone_tar_manifest, extract_zone_tar, write_compact
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True)
t0 = time.time()
for z in zones:
    for src, dst in ((f'{CACHE_DRIVE}/{z}/points.npz', f'/content/cache/{z}/points.npz'), (f'{SUBSET_DIR}/{z}/subset.npz', f'/content/banks/{z}/subset.npz')):
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
print(f"LiDAR caches + feature banks of {len(zones)} zones ({(time.time() - t0) / 60:.1f} min)")
CACHE2D = '/content/cache2d'; t0 = time.time(); slow = []
for z in zones:
    if zone_tar_manifest(CACHE2D_DRIVE, z, wait=30) is not None:   # one big sequential read: fast
        n = extract_zone_tar(CACHE2D_DRIVE, z, CACHE2D); how = 'tar'
    else:                                                             # no tar yet: parallel per-file copy (slow)
        ids = cached_image_ids(CACHE2D_DRIVE, z)
        with ThreadPoolExecutor(16) as ex:
            list(ex.map(lambda i: write_compact(CACHE2D_DRIVE, z, i, CACHE2D), ids))
        n = len(cached_image_ids(CACHE2D, z)); how = 'files'; slow.append(z)
    print(f"  {z}: {n} images from {how}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
if slow: print("zones without a tar (slow path) - run colab_compact_cache2d once for:", ','.join(slow))
code = ("import sys; sys.path.insert(0,'/content/code'); from gridnet.train.data2d import fetch_zones; "
        f"fetch_zones({zones!r}, '{DATA}', what=('pose',))")
subprocess.run([PY, '-c', code], check=True); print("camera files ready")
print("associations present:", [z for z in zones if os.path.exists(f'{ASSOC_DIR}/{z}/assoc.npz')], "(missing ones are computed on first use)")
print(subprocess.run(['df', '-h', '--output=avail', '/content'], capture_output=True, text=True).stdout.split()[-1], "free on the local disk")


In [ ]:
#@title 6. Protocol, draw 0 of LEVELS only, with the predictions around the pylons saved -> GridNet/qualitative/<TAG>/ (resumes per zone)
import subprocess, time, os
cmd = [PY, '/content/code/scripts/run_protocol.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
       '--assoc-dir', ASSOC_DIR, '--out', f'{OUT}/protocol', '--subset-dir', '/content/banks', '--litept', '/content/LitePT',
       '--zones', ','.join(zones), '--sweeps', ','.join(sweeps), '--draws', '1', '--mode', MODE, '--seed', str(SEED),
       '--dump-pred', OUT, '--dump-levels', LEVELS, '--dump-radius', str(RADIUS), '--dump-pylons', str(PYLONS)]
for sw, lv in sweeps.items(): cmd += [f'--levels-{sw}', ','.join(f'{x:g}' for x in lv)]
cmd += ['--heads'] + [f'{n}={p}' for n, p in heads.items()]
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True', MPLBACKEND='Agg')
t0 = time.time(); tail = []
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
try:
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if line.startswith('[protocol]'): print(f"   {(time.time() - t0) / 60:5.1f} min  {line}", end='', flush=True)
        elif 'Error' in line or 'Traceback' in line or line.startswith('  File'): print(line, end='')
except KeyboardInterrupt:
    p.terminate(); p.wait(); print("stopped by user; finished zones are on Drive -> run this cell again to continue"); raise
p.wait(); print(f"exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
if p.returncode != 0:
    print(''.join(l for l in tail if not l.startswith('[protocol]'))[-3000:])
    print("-> if it STOPPED on 'other settings': this TAG holds a run with other heads / levels / radius / pylons / mode - give TAG a new name in cell 2, re-run cell 2 and this cell")
    print("-> if it ran out of GPU memory on a GPU smaller than the A100: set MODE = 'fast' and a new TAG in cell 2")
crops = sorted(f for f in os.listdir(OUT) if f.endswith('.npz') and '_crop' in f and '.tmp' not in f)
print(f"{len(crops)} crop files on Drive:", ', '.join(f"{f} ({os.path.getsize(f'{OUT}/{f}') / 2**20:.1f} MB)" for f in crops))
assert p.returncode == 0 and crops, "the run did not finish: see the lines above"


In [ ]:
#@title 7. Check against the benchmark run (same cameras -> same confusion matrices), table of the crops, previews
import subprocess, json, os
from IPython.display import Image, display
env = dict(os.environ, PYTHONPATH='/content/code', MPLBACKEND='Agg')
ref = f'{DRIVE}/protocol/{REFERENCE_RUN}'
if REFERENCE_RUN and (os.path.exists(f'{ref}/results.json') or os.path.exists(f'{ref}/results.json.gz')):
    r = subprocess.run([PY, '/content/code/scripts/compare_protocol.py', '--a', f'{OUT}/protocol', '--b', ref, '--names', f'figure,{REFERENCE_RUN}',
                        '--out', f'{OUT}/compare_{REFERENCE_RUN}.json'], capture_output=True, text=True, env=env)
    print(r.stdout[-9000:], r.stderr[-1500:])
    if r.returncode == 0:
        rows = json.load(open(f'{OUT}/compare_{REFERENCE_RUN}.json'))['rows']
        worst = max(max(abs(x['mean_all_b'] - x['mean_all_a']), abs(x['mean_electrical_b'] - x['mean_electrical_a'])) for x in rows)
        verdict = ("(fast mode: small differences to the exact run are expected)" if MODE != 'exact' else
                   "-> the crops show predictions of the benchmark" if 100 * worst < 0.05 else "-> NOT the same predictions: find the cause before the figure is used")
        print(f"largest difference to {REFERENCE_RUN} over {len(rows)} (model, level) cells: {100 * worst:.3f} points of mean IoU {verdict}")
else:
    print(f"no reference run at {ref}: comparison skipped")
r = subprocess.run([PY, '/content/code/scripts/plot_qualitative.py', '--list', OUT], capture_output=True, text=True, env=env)
print(r.stdout[-14000:], r.stderr[-1500:])
open(f'{OUT}/crops.txt', 'w').write(r.stdout)
os.makedirs(f'{OUT}/preview', exist_ok=True)
for f in sorted(x for x in os.listdir(OUT) if x.endswith('.npz') and '_crop' in x and '.tmp' not in x):
    base = f"{OUT}/preview/{f[:-4]}"
    r = subprocess.run([PY, '/content/code/scripts/plot_qualitative.py', '--crop', f'{OUT}/{f}', '--models', PREVIEW_MODELS, '--levels', PREVIEW_LEVELS,
                        '--out', base], capture_output=True, text=True, env=env)
    print(r.stdout.strip() or r.stderr[-1500:])
    if os.path.exists(base + '.png'): display(Image(base + '.png', width=1000))


In [ ]:
#@title 8. (optional) Shut the runtime down once everything above has finished - saves compute units on unattended runs
if SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    print("flushing pending writes to Drive ..."); drive.flush_and_unmount()
    print("results are on Drive; shutting the runtime down"); runtime.unassign()
else:
    print("SHUTDOWN_WHEN_DONE is off -> remember to Disconnect and delete runtime")


## หมายเหตุ
- ไฟล์ crop หนึ่งไฟล์ = เสาไฟฟ้าหนึ่งต้น: จุดของ subset (จุดของ 4 electrical classes ครบตาม cap ของ subset + ตัวอย่างของ class อื่น) ภายในรัศมี `RADIUS` ม.
  ถูกลดเหลือหนึ่งจุดต่อ voxel 5 ซม. สำหรับ **วาดรูป** ส่วน **สถิติ** ของ crop (confusion matrix ในไฟล์) คิดจากจุดของ subset ทุกจุดในรัศมีพร้อม weight
  จึงเป็นตัวเลขของ benchmark ที่จำกัดเฉพาะบริเวณนั้น — พื้นและต้นไม้รอบ ๆ เป็นแค่ตัวอย่างเบาบาง (ไม่ใช่ cloud เต็ม) รูปนี้จึงเป็นรูปของสายและเสา
- `LEVELS` = ระดับที่เก็บ: pose ที่ให้มา, 10 cm (reference level: fusion heads ไม่ควรเปลี่ยน แต่ ImageVote เริ่มเสียสาย), 50 cm และ 1° (ระดับใหญ่สุดของตาราง)
- รูป preview ของ cell 7: แถว = `PREVIEW_MODELS`, คอลัมน์ = `PREVIEW_LEVELS`; จุดของสาย/เสาที่ทำนาย class ถูก = เทา, ผิด = แดง, จุดรอบ ๆ ที่ถูกทำนายเป็นสาย/เสา = น้ำเงิน;
  ตัวเลขในแต่ละช่อง = สัดส่วนจุดของ electrical classes ใน crop ที่ทำนายผิด (ถ่วง weight) — จุดผิดถูกวาดทับและขยาย พื้นที่สีแดงในรูปจึงดูมากกว่าตัวเลข
- เสาถูกเลือกจากจำนวนจุดของ class pylon (ต้นที่ใหญ่ที่สุดของแต่ละโซน) ก่อนเห็น prediction ใด ๆ; การเลือกว่าจะใช้ต้นไหนใน paper ทำทีหลังจาก `crops.txt`
  และ caption ต้องบอกว่าเลือกอย่างไร
- `drift` ใน `LEVELS` ได้เช่นกัน (เช่น `drift:0.5`): draw 0 = ทิศ +
- จบแล้ว **Disconnect and delete runtime**